# Hybrid Search and reranking in RAG

In [ ]:
!pip install weaviate-client

In [ ]:
!pip install langchain langchain_community

In [ ]:
# Paste your Weaviate cluster URL
import weaviate
WEAVIATE_CLUSTER = ""

In [ ]:
WEAVIATE_URL = WEAVIATE_CLUSTER
# Replace 'YOUR_WEAVIATE_API_KEY' with your actual Weaviate API key, or load it from Colab secrets/environment variables.
# Example: WEAVIATE_API_KEY = os.getenv("WEAVIATE_API_KEY")
WEAVIATE_API_KEY = "YOUR_WEAVIATE_API_KEY"

In [ ]:
# Paste your Hugging Face token
HF_TOKEN = ""

In [ ]:
import os
import weaviate
from weaviate.auth import AuthApiKey

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=AuthApiKey(WEAVIATE_API_KEY),
    headers={
         "X-HuggingFace-Api-Key": HF_TOKEN
    }
)

In [ ]:
client.is_ready()

In [ ]:
collections_dict = client.collections.list_all()
if collections_dict:
    print("Existing Collections (Classes):")
    for collection_obj in collections_dict.values():
        print(f"- Name: {collection_obj.name}")
        # To get the detailed configuration, fetch the full Collection object
        full_collection_details = client.collections.get(collection_obj.name)
        if full_collection_details:
            print(f"  Config: {full_collection_details.config.get()}")
        else:
            print(f"  Could not retrieve detailed config for {collection_obj.name}")
else:
    print("No collections (classes) found in the Weaviate instance.")

In [ ]:
# In Weaviate Python Client v4, to delete all collections, you need to iterate and delete them individually.
for collection_name in client.collections.list_all():
    client.collections.delete(collection_name)
print("All collections deleted successfully.")

In [ ]:
schema = {
    "classes": [
        {
            "class": "RAG",
            "description": "Documents for RAG",
            "vectorizer": "text2vec-huggingface",
            "moduleConfig": {"text2vec-huggingface": {"model": "sentence-transformers/all-MiniLM-L6-v2", "type": "text"}},
            "properties": [
                {
                    "dataType": ["text"],
                    "description": "The content of the paragraph",
                    "moduleConfig": {
                        "text2vec-huggingface": {
                            "skip": False,
                            "vectorizePropertyName": False,
                        }
                    },
                    "name": "content",
                },
            ],
        },
    ]
}

In [ ]:
import weaviate.classes.config as wc

# Get the class definition from the existing schema dictionary
rag_class_def = schema["classes"][0]

# Define vectorizer configuration for v4
vectorizer_config_v4 = wc.Configure.Vectorizer.text2vec_huggingface(
    model=rag_class_def["moduleConfig"]["text2vec-huggingface"]["model"]
    # The 'type' parameter is not used in v4's text2vec_huggingface configuration
)

# Define properties for v4
properties_v4 = []
for prop_def in rag_class_def["properties"]:
    properties_v4.append(
        wc.Property(
            name=prop_def["name"],
            data_type=wc.DataType.TEXT, # Assuming 'text' for all data types in this example schema
            description=prop_def["description"],
            skip_vectorization=prop_def["moduleConfig"]["text2vec-huggingface"]["skip"],
            vectorize_property_name=prop_def["moduleConfig"]["text2vec-huggingface"]["vectorizePropertyName"]
        )
    )

# Create the collection using v4 client method
client.collections.create(
    name=rag_class_def["class"],
    description=rag_class_def["description"],
    vectorizer_config=vectorizer_config_v4,
    properties=properties_v4,
)

print(f"Collection '{rag_class_def['class']}' created successfully.")

In [ ]:
# In Weaviate Python Client v4, schema operations are done via client.collections
collections_dict = client.collections.list_all()
if collections_dict:
    print("Existing Collections (Classes):")
    for collection_obj in collections_dict.values():
        print(f"- Name: {collection_obj.name}")
        # To get the detailed configuration, fetch the full Collection object
        full_collection_details = client.collections.get(collection_obj.name)
        if full_collection_details:
            print(f"  Config: {full_collection_details.config.get()}")
        else:
            print(f"  Could not retrieve detailed config for {collection_obj.name}")
else:
    print("No collections (classes) found in the Weaviate instance.")

In [ ]:
!pip install langchain-weaviate

In [ ]:
from langchain_weaviate import WeaviateVectorStore
from langchain_community.embeddings import HuggingFaceEmbeddings

# Instantiate embedding model
# Use the same model as configured in Weaviate for text2vec-huggingface
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)

# Instantiate WeaviateVectorStore with the v4 client
vector_store = WeaviateVectorStore(
    client=client,
    index_name="RAG",
    text_key="content",
    attributes=[],
    embedding=embeddings, # Pass the embedding model here
)

# Convert the vector store to a retriever.
# Configure hybrid search using search_kwargs in as_retriever()
retriever = vector_store.as_retriever(
    search_kwargs={"alpha": 0.5} # Pass alpha for hybrid search here
)

In [ ]:
# The previous model 'HuggingFaceH4/zephyr-7b-beta' is a 7B model and might exceed available RAM.
# Let's try a smaller model like TinyLlama (1.1B parameters) to avoid out-of-memory issues.
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# You may need to restart the runtime after changing the model name to free up memory from the previous model.

In [ ]:
!pip install bitsandbytes
!pip install accelerate

In [ ]:
import torch
from transformers import ( AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline, )
from langchain_community.llms import HuggingFacePipeline

In [ ]:
# Function for loading 4-bit quantized model
def load_quantized_model(model_name: str):
    """
    model_name: Name or path of the model to be loaded.
    return: Loaded quantized model.
    """
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        low_cpu_mem_usage=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.bfloat16, # Changed torch_dtype to dtype
        quantization_config=bnb_config,
    )
    return model

In [ ]:
# Initializing tokenizer
def initialize_tokenizer(model_name: str):
    """
    model_name: Name or path of the model for tokenizer initialization.
    return: Initialized tokenizer.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name, return_token_type_ids=False)
    tokenizer.bos_token_id = 1  # Set beginning of sentence token id
    return tokenizer

In [ ]:
tokenizer = initialize_tokenizer(model_name)

In [ ]:
model = load_quantized_model(model_name)

In [ ]:
pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    use_cache=True,
    device_map="auto",
    #max_length=2048,
    do_sample=True,
    top_k=5,
    max_new_tokens=100,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

In [ ]:
llm = HuggingFacePipeline(pipeline=pipeline)

In [ ]:
doc_path="/content/RS1.pdf"

In [ ]:
!pip install pypdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
loader = PyPDFLoader(doc_path)

In [ ]:
docs = loader.load()

In [ ]:
docs

In [ ]:
from langchain_core.documents import Document

# Clean metadata from documents before adding to Weaviate
# Only the 'content' property is defined in the Weaviate schema for 'RAG'.
# Other metadata fields from PyPDFLoader (e.g., 'ptex.fullbanner') cause ingestion errors.
cleaned_docs = []
for doc in docs:
    cleaned_docs.append(Document(page_content=doc.page_content, metadata={}))

retriever.add_documents(cleaned_docs)

In [ ]:
result = retriever.invoke("What is RAG token?")

In [ ]:
result

In [ ]:
result[0]

In [ ]:
len(retriever.invoke("What is RAG token"))

In [ ]:
from langchain.chains import RetrievalQA

In [ ]:
hybrid_chain = RetrievalQA.from_chain_type(llm = llm, chain_type = "stuff", retriever = retriever)

In [ ]:
result = hybrid_chain.invoke("What is RAG token?")

In [ ]:
result
print(result)

In [ ]:
!pip install cohere

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CohereRerank

In [ ]:
compressor = CohereRerank(cohere_api_key = "")

In [ ]:
compression_retriever = ContextualCompressionRetriever(
	base_compressor = compressor, base_retriever = retriever
)

In [ ]:
compressed_docs = compression_retriever.get_relevant_documents("What is RAG token?")
print(compressed_docs[0])

In [ ]:
hybrid_chain = RetrievalQA.from_chain_type(llm = llm, chain_type = "stuff", retriever = compression_retriever)

In [ ]:
response = hybrid_chain.invoke("What is Abstractive Question Answering?")

In [ ]:
print(response["result"])